# Indexing

## 1. Load

In [1]:
import json

In [2]:
domains = json.load(open("domain.json", "r"))

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

# model_name = "dunzhang/stella_en_1.5B_v5"
# model_name = "dunzhang/stella_en_400M_v5"
model_name = "sentence-transformers/all-mpnet-base-v2"
model_kwargs = {'device': 'cpu'}
encode_kwargs = {'normalize_embeddings': False}

embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

In [5]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

index = faiss.IndexFlatL2(len(embeddings.embed_query("matchhub")))

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

## 2. Split

In [2]:
import faiss
import json
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

# model_name = "dunzhang/stella_en_1.5B_v5"
# model_name = "dunzhang/stella_en_400M_v5"
model_name = "sentence-transformers/all-mpnet-base-v2"
model_kwargs = {'device': 'cpu'}
encode_kwargs = {'normalize_embeddings': False}

embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

2024-12-26 23:27:53.098203: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-12-26 23:27:53.106722: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1735243073.117443   16963 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1735243073.120452   16963 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-12-26 23:27:53.131679: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [4]:
domains_intents = json.load(open("domain.json", "r"))

index = faiss.IndexFlatL2(len(embeddings.embed_query("MatchHub")))

vector_store = FAISS(
	embedding_function=embeddings,
	index=index,
	docstore=InMemoryDocstore(),
	index_to_docstore_id={},
)

for domain in domains_intents.keys():
	for intent in domains_intents[domain].keys():
		examples = domains_intents[domain][intent]
		if len(examples) > 1:
			print(f"Adding examples of domain {domain} and intent {intent} with {len(examples)} examples")
			for i, example in enumerate(examples):
				document = Document(
					page_content=example,
					metadata = {
						"domain": domain,
						"intent": intent
					}
				)
				vector_store.add_documents(documents=[document], ids=[f"{domain}_{intent}_{i}"])

Adding examples of domain T-shirts and intent None with 30 examples
Adding examples of domain T-shirts and intent Buying with 30 examples
Adding examples of domain T-shirts and intent Selling with 30 examples
Adding examples of domain Polo shirts and intent None with 30 examples
Adding examples of domain Polo shirts and intent Buying with 30 examples
Adding examples of domain Polo shirts and intent Selling with 30 examples
Adding examples of domain Blouses and intent None with 30 examples
Adding examples of domain Blouses and intent Buying with 30 examples
Adding examples of domain Blouses and intent Selling with 30 examples
Adding examples of domain Dress shirts and intent None with 29 examples
Adding examples of domain Dress shirts and intent Buying with 30 examples
Adding examples of domain Dress shirts and intent Selling with 30 examples
Adding examples of domain Tank tops and intent None with 29 examples
Adding examples of domain Tank tops and intent Buying with 30 examples
Adding

In [4]:
vector_store.delete(ids=[f"{domain}_{intent}_{i}"])

True

In [ ]:
vector_store.save_local("domain_intent")

## 3. Store

In [ ]:
vector_store.save_local("domain_intent")

In [5]:
vector_store = FAISS.load_local(
    "domain_intent", embeddings, allow_dangerous_deserialization=True
)

# Retrieval and Generation

In [4]:
results = vector_store.similarity_search(
    "LangChain provides abstractions to make working with LLMs easy",
    k=2,
    filter={"source": "tweet"},
)
for res in results:
    print(f"* {res.page_content} [{res.metadata}]")

In [9]:
len(results)

10

In [10]:
# prompt = "Polo T-shirts for men on sale, suitable for semi-formal occasions."
prompt = "I need to buy some T-shirts"

In [13]:
results = vector_store.similarity_search_with_score(
    prompt, k=100, 
    filter=None
)
for res, score in results:
    print(res.metadata)
    print(f"* [SIM={score:3f}] {res.page_content} [{res.metadata}]")
    print()

{'domain': 'T-shirts', 'intent': 'Selling'}
* [SIM=0.604153] Affordable men’s T-shirts starting at $10, bulk orders available. [{'domain': 'T-shirts', 'intent': 'Selling'}]

{'domain': 'T-shirts', 'intent': 'Selling'}
* [SIM=0.672805] Affordable women’s T-shirts starting at $8, available in bulk. [{'domain': 'T-shirts', 'intent': 'Selling'}]

{'domain': 'T-shirts', 'intent': 'Buying'}
* [SIM=0.725911] I need oversized T-shirts for a comfortable fit. [{'domain': 'T-shirts', 'intent': 'Buying'}]

{'domain': 'Polo shirts', 'intent': 'Selling'}
* [SIM=0.739619] Branded polo shirts like Ralph Lauren and Lacoste for sale. [{'domain': 'Polo shirts', 'intent': 'Selling'}]

{'domain': 'T-shirts', 'intent': 'Buying'}
* [SIM=0.758222] Do you stock Calvin Klein or Tommy Hilfiger T-shirts? [{'domain': 'T-shirts', 'intent': 'Buying'}]

{'domain': 'Polo shirts', 'intent': 'Selling'}
* [SIM=0.762451] Classic striped white and navy polo shirts in stock. [{'domain': 'Polo shirts', 'intent': 'Selling'}]


In [ ]:
retriever = vector_store.as_retriever(search_type="mmr", search_kwargs={"k": 1})
retriever.invoke("Stealing from the bank is a crime", filter={"source": "news"})

In [ ]:
vector_store.save_local("faiss_index")

new_vector_store = FAISS.load_local(
    "faiss_index", embeddings, allow_dangerous_deserialization=True
)

docs = new_vector_store.similarity_search("qux")

## 4. Retrieve

## 5. Generate